In [ ]:
import os
import sys

import torch
import transformers
from torch.amp import autocast
from transformers import (
    BitsAndBytesConfig,
    GroundingDinoForObjectDetection,
    GroundingDinoProcessor,
)

from utils.utils import device, pic2float

print("python:", sys.executable)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

# добавить папку уровнем выше (корень проекта) в PYTHONPATH
parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [ ]:
def preprocess_caption(caption: str) -> str:
    result = caption.lower().strip()
    if result.endswith("."):
        return result
    return result + "."


class GDINO:
    def __init__(self, model_id="IDEA-Research/grounding-dino-tiny", device="cuda"):
        self.device = device

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,  # включаем 4-битную квантизацию
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=False,
            bnb_4bit_quant_type="nf4",  # можно попробовать и другие варианты, например 'fp4'
        )

        self.processor = GroundingDinoProcessor.from_pretrained(model_id)
        self.model = GroundingDinoForObjectDetection.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            quantization_config=quant_config,
            low_cpu_mem_usage=True,
        ).to(device)

    def detect_objects(self, image, text="a cat"):
        image = pic2float(image)
        text = preprocess_caption(text)
        inputs = self.processor(image, return_tensors="pt", text=text, do_rescale=False).to(
            self.device
        )

        with torch.no_grad():
            with autocast("cuda", dtype=torch.float16):
                outputs = self.model(**inputs)

        results = self.processor.image_processor.post_process_object_detection(
            outputs, target_sizes=[image.shape[:-1]], threshold=0.1
        )[0]
        return results


print("Loading GDINO model...")
gdino = GDINO()
print("GDINO model loaded")

In [ ]:
model_id = "IDEA-Research/grounding-dino-tiny"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,  # включаем 4-битную квантизацию
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",  # можно попробовать и другие варианты, например 'fp4'
)

processor = GroundingDinoProcessor.from_pretrained(model_id)

model = GroundingDinoForObjectDetection.from_pretrained(model_id)

In [ ]:
model_id = "IDEA-Research/grounding-dino-tiny"

print("loading processor...")
processor = GroundingDinoProcessor.from_pretrained(model_id, local_files_only=True)
print("processor ok")

print("loading model...")
model = GroundingDinoForObjectDetection.from_pretrained(model_id, local_files_only=True)
print("model loaded on CPU")

model = model.to(device)
print("moved to", device)

loading processor...
processor ok
loading model...
model loaded on CPU
moved to cuda
